# Preparation

In [ ]:
import re
import ipykernel
from notebook.notebookapp import list_running_servers
import requests
from urllib.parse import urljoin
import json
import os
from google.colab import auth
from google.colab import userdata
from googleapiclient.discovery import build
import pandas as pd

Change the working directory to the path of the current notebook

In [ ]:
from google.colab import drive
# Mount Google Drive to access files stored in it
drive.mount('/content/drive')


def get_notebook_path():
    # Get the path of the current Jupyter notebook
    kernel_id = re.search('kernel-(.*).json', ipykernel.get_connection_file()).group(1)
    servers = list_running_servers()
    for ss in servers:
        response = requests.get(urljoin(ss['url'], 'api/sessions'), params={'token': ss.get('token', '')})
        for nn in json.loads(response.text):
            if nn['kernel']['id'] == kernel_id:
                relative_path = nn['notebook']['path']
                return os.path.join(ss['notebook_dir'], relative_path)

def get_folder_path(folder_id):
    # Recursively get the full path of a folder given its ID
    if folder_id:
        folder = drive_service.files().get(fileId=folder_id, fields="name, parents").execute()
        folder_name = folder.get('name')
        parents = folder.get('parents')
        if parents:
            parent_path = get_folder_path(parents[0])
            return parent_path + '/' + folder_name
        else:
            return folder_name
    return ''

def get_file_path(file_id):
    # Recursively get the full path of a file given its ID
    file = drive_service.files().get(fileId=file_id, fields="name, parents").execute()
    file_name = file.get('name')
    parents = file.get('parents')
    if parents:
        parent_id = parents[0]
        parent_path = get_folder_path(parent_id)
        return parent_path
    else:
        return file_name

# Get the path of the current notebook
notebook_path = get_notebook_path()


# Authenticate and initialize the Google Drive API
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Extract the file ID from the notebook path (assumes file ID is part of the path)
file_id = re.search(r'fileId=(\w+)', notebook_path).group(1)

# Get the full path of the file using its ID
file_path = get_file_path(file_id)
if 'マイドライブ' in file_path:
    converted_path = re.sub(r'(^|/)マイドライブ($|/)', '/content/drive/MyDrive/', file_path)
elif 'MyDrive' in file_path:
    converted_path = re.sub(r'(^|/)MyDrive($|/)', '/content/drive/MyDrive/', file_path)
else:
    converted_path = '/content/drive/MyDrive/' + file_path

# Change the working directory to the converted path
os.chdir(converted_path)

In [ ]:
#function to calculate sensitivity
def sensitivity_calculator(group, result_cols):
    sensitivity_numerator = 0
    sensitivity_denominator = 0
    for col in result_cols:
        # If a value manually extracted by a human exists
        if not(pd.isna(group.iloc[1][col]) or group.iloc[1][col] in ["", "*", "NaN"]):
            sensitivity_denominator += 1
            # If GPT's extraction matches the human's extraction
            if group.iloc[2][col] == "True":
                sensitivity_numerator += 1
            elif float(group.iloc[2][col]) == 1.0:
                sensitivity_numerator += 1
            # If GPT's extraction does not match the human's
            elif group.iloc[2][col] == "False":
                continue;
            # If the value is NA
            elif group.iloc[2][col] in ["NA", np.nan]:
                sensitivity_denominator -= 1 # Do not count as target present
    return sensitivity_numerator, sensitivity_denominator

#function to calculate specificity
def specificity_calculator(group, result_cols):
    specificity_numerator = 0
    specificity_denominator = 0
    for col in result_cols:
        # If a value manually not extracted by a human exists
        if pd.isna(group.iloc[1][col]) or group.iloc[1][col] in ["", "*", "NaN"]:
            specificity_denominator += 1
            # If GPT's extraction matches the human's extraction
            if group.iloc[2][col] == "True":
                specificity_numerator += 1
            elif float(group.iloc[2][col]) == 1.0:
                specificity_numerator += 1
            # If GPT's extraction does not match the human's
            elif group.iloc[2][col] == "False":
                continue;
            # If the value is NA
            elif group.iloc[2][col] in ["NA", np.nan]:
                specificity_denominator -= 1 # Do not count as target present
    return specificity_numerator, specificity_denominator

#function to calculate precision
def precision_calculator(group, result_cols):

    precision_numerator = 0
    precision_denominator = 0

    for col in result_cols:
        if not((str(group.iloc[0][col]) == "-1") or (str(group.iloc[0][col]) == "null")):
            precision_denominator += 1
            if group.iloc[2][col] == "True":
                precision_numerator += 1
            elif group.iloc[2][col] == "False":
                continue;
            elif float(group.iloc[2][col]) == 1.0:
                precision_numerator += 1
            elif group.iloc[2][col] in ["NA", np.nan]:
                precision_denominator -= 1

    return precision_numerator, precision_denominator


# function to calculate sensitivity for each variable
def sensitivity_calculator_variable(group, result_cols):
    sensitivity_numerator = [0] * len(result_cols)
    sensitivity_denominator = [0] * len(result_cols)
    for i, col in enumerate(result_cols):
        if not(pd.isna(group.iloc[1][col]) or group.iloc[1][col] in ["", "*", "NaN"]):
            sensitivity_denominator[i] += 1
            if group.iloc[2][col] == "True":
                sensitivity_numerator[i] += 1
            elif group.iloc[2][col] in ["NA", np.nan]:
                sensitivity_denominator[i] -= 1
    return sensitivity_numerator, sensitivity_denominator

# function to calculate specificity for each variable
def specificity_calculator_variable(group, result_cols):
    specificity_numerator = [0] * len(result_cols)
    specificity_denominator = [0] * len(result_cols)
    for i, col in enumerate(result_cols):
        if pd.isna(group.iloc[1][col]) or group.iloc[1][col] in ["", "*", "NaN"]:
            specificity_denominator[i] += 1
            if group.iloc[2][col] == "True":
                specificity_numerator[i] += 1
            elif group.iloc[2][col] in ["NA", np.nan]:
                specificity_denominator[i] -= 1
    return specificity_numerator, specificity_denominator

# function to calculate precision for each variable
def precision_calculator_variable(group, result_cols):
    precision_numerator = [0] * len(result_cols)
    precision_denominator = [0] * len(result_cols)
    for i, col in enumerate(result_cols):
        if not((str(group.iloc[0][col]) == "-1") or (str(group.iloc[0][col]) == "null")):
            precision_denominator[i] += 1
            if group.iloc[2][col] == "True":
                precision_numerator[i] += 1
            elif group.iloc[2][col] in ["NA", np.nan]:
                precision_denominator[i] -= 1
    return precision_numerator, precision_denominator


# calculate each metric for fold
def result_calculator_for_each(n_idx_min, n_idx_max, accuracy_numerator_list, accuracy_denominator_list, sensitivity_numerator_list, sensitivity_denominator_list, specificity_numerator_list, specificity_denominator_list, precision_numerator_list, precision_denominator_list):
  results_for_fold = []
  for n_idx in range(n_idx_min, n_idx_max+1):
      for fold in range(10):
          accuracy = accuracy_numerator_list[n_idx][fold] / accuracy_denominator_list[n_idx][fold] if accuracy_denominator_list[n_idx][fold] != 0 else np.nan
          sensitivity = sensitivity_numerator_list[n_idx][fold] / sensitivity_denominator_list[n_idx][fold] if sensitivity_denominator_list[n_idx][fold] != 0 else np.nan
          specificity = specificity_numerator_list[n_idx][fold] / specificity_denominator_list[n_idx][fold] if specificity_denominator_list[n_idx][fold] != 0 else np.nan
          precision = precision_numerator_list[n_idx][fold] / precision_denominator_list[n_idx][fold] if precision_denominator_list[n_idx][fold] != 0 else np.nan

          results_for_fold.append({
              "n_index": n_idx,
              "fold_index": fold,
              "accuracy": accuracy,
              "sensitivity": sensitivity,
              "specificity": specificity,
              "precision": precision
          })
  return results_for_fold

# Calculate the average and standard deviation of metrics across folds
def result_calculator_all(n_idx_min, n_idx_max,results_for_fold):
  result_all = []

  # Loop over each n_index in the specified range
  for n_idx in range(n_idx_min, n_idx_max+1):
        accuracy_list = []
        sensitivity_list = []
        specificity_list = []
        precision_list = []

        # Collect metrics from all 10 folds for the current n_index
        for fold in range(10):
            result = next((r for r in results_for_fold if r["n_index"] == n_idx and r["fold_index"] == fold), None)
            if result is not None:
                # Append each metric to the corresponding list if the value is not NaN
                if not np.isnan(result["accuracy"]):
                    accuracy_list.append(result["accuracy"])
                if not np.isnan(result["sensitivity"]):
                    sensitivity_list.append(result["sensitivity"])
                if not np.isnan(result["specificity"]):
                    specificity_list.append(result["specificity"])
                if not np.isnan(result["precision"]):
                    precision_list.append(result["precision"])

        # Compute mean of each metric
        accuracy_avg = np.nanmean(accuracy_list) if accuracy_list else np.nan
        sensitivity_avg = np.nanmean(sensitivity_list) if sensitivity_list else np.nan
        specificity_avg = np.nanmean(specificity_list) if specificity_list else np.nan
        precision_avg = np.nanmean(precision_list) if precision_list else np.nan

        # Compute standard deviation
        accuracy_std = np.nanstd(accuracy_list, ddof=1) if len(accuracy_list) > 1 else np.nan
        sensitivity_std = np.nanstd(sensitivity_list, ddof=1) if len(sensitivity_list) > 1 else np.nan
        specificity_std = np.nanstd(specificity_list, ddof=1) if len(specificity_list) > 1 else np.nan
        precision_std = np.nanstd(precision_list, ddof=1) if len(precision_list) > 1 else np.nan

        # Store the averaged metrics and their standard deviations (scaled to percentages)
        result_all.append({
            "n_index": n_idx,
            "accuracy_mean": accuracy_avg*100,
            "accuracy_std": accuracy_std*100,
            "sensitivity_mean": sensitivity_avg*100,
            "sensitivity_std": sensitivity_std*100,
            "specificity_mean": specificity_avg*100,
            "specificity_std": specificity_std*100,
            "precision_mean": precision_avg*100,
            "precision_std": precision_std*100
        })
  return result_all

#Calculate metrics for all variables

In [ ]:
import pandas as pd
import numpy as np

# Load the Excel file (first sheet)
file_name = "analysis_contextual_chat_prompting"
file_path = file_name + ".xlsx"
df = pd.read_excel(file_path, sheet_name=0)


# Fill NaN values in 'n_index' and 'fold_index'
df[["n_index", "fold_index"]] = df[["n_index", "fold_index"]].fillna(method="ffill")

# Identify groups of every 4 rows corresponding to one arm
df["group_index"] = df.index // 4

# Identify the columns that contain True / False results
result_cols = df.iloc[:, 6:-1].columns

# Initialize lists to store numerators and denominators for each metric (6 n_index values × 10 folds)
accuracy_numerator_list = [[0] * 10 for _ in range(6)]
accuracy_denominator_list = [[0] * 10 for _ in range(6)]
sensitivity_numerator_list = [[0] * 10 for _ in range(6)]
sensitivity_denominator_list = [[0] * 10 for _ in range(6)]
specificity_numerator_list = [[0] * 10 for _ in range(6)]
specificity_denominator_list = [[0] * 10 for _ in range(6)]
precision_numerator_list = [[0] * 10 for _ in range(6)]
precision_denominator_list = [[0] * 10 for _ in range(6)]

# Group the DataFrame by n_index, fold_index, and group_index
grouped = df.groupby(["n_index", "fold_index", "group_index"])

for (n_idx, fold, group_idx), group in grouped:
    # Identify the Standard_Ref row
    std_ref_col = df.loc[group.index[1], result_cols] if len(group) > 1 else None

    # Calculate Accuracy: count "True" and "False"
    true_count = (group[result_cols] == "True").sum().sum()
    false_count = (group[result_cols] == "False").sum().sum()
    accuracy_numerator_list[int(n_idx)][int(fold)] += true_count
    accuracy_denominator_list[int(n_idx)][int(fold)] += true_count + false_count

    # Calculate Sensitivity
    sensitivity_numerator, sensitivity_denominator = sensitivity_calculator(group, result_cols)
    sensitivity_numerator_list[int(n_idx)][int(fold)] += sensitivity_numerator
    sensitivity_denominator_list[int(n_idx)][int(fold)] += sensitivity_denominator

    # Calculate Specificity
    specificity_numerator, specificity_denominator = specificity_calculator(group, result_cols)
    specificity_numerator_list[int(n_idx)][int(fold)] += specificity_numerator
    specificity_denominator_list[int(n_idx)][int(fold)] += specificity_denominator

    # Calculate precision
    precision_numerator, precision_denominator = precision_calculator(group, result_cols)
    precision_numerator_list[int(n_idx)][int(fold)] += precision_numerator
    precision_denominator_list[int(n_idx)][int(fold)] += precision_denominator

results_for_fold = result_calculator_for_each(0,5,accuracy_numerator_list, accuracy_denominator_list, sensitivity_numerator_list, sensitivity_denominator_list, specificity_numerator_list, specificity_denominator_list, precision_numerator_list, precision_denominator_list)
result_all = result_calculator_all(0,5,results_for_fold)

#Save result_for_each and result_all into separate sheets in an Excel file.
df_for_each = pd.DataFrame(results_for_fold)
df_all = pd.DataFrame(result_all)
with pd.ExcelWriter("new_sensitivity_specificity/" + file_name+"_new_sensitivity_specificity.xlsx") as writer:
    df_for_each.to_excel(writer, sheet_name="result_for_each", index=False)
    df_all.to_excel(writer, sheet_name="result_all", index=False)precision_

<ipython-input-25-380c25a0bbc0>:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[["n_index", "fold_index"]] = df[["n_index", "fold_index"]].fillna(method="ffill")


#Calculate metrics for all varibles and for each variable


## numeric

In [ ]:
from pickle import TRUE
import pandas as pd
import numpy as np
import copy

# Load the Excel file (first sheet)
file_name = "analysis_contextual_chat_prompting_after_modification"
file_path = file_name + ".xlsx"
df = pd.read_excel(file_path, sheet_name=0)
df = df[:303].reset_index(drop=True)

# Fill NaN values in 'n_index' and 'fold_index'
df[["n_index", "fold_index"]] = df[["n_index", "fold_index"]].fillna(method="ffill")

# Identify groups of every 4 rows corresponding to one arm
df["group_index"] = df.index // 4

# Identify the columns that contain True / False results
result_cols = df.iloc[:, 6:-1].columns

accuracy_numerator_list = [[0] * 10 for _ in range(6)]
accuracy_denominator_list = [[0] * 10 for _ in range(6)]
sensitivity_numerator_list = [[0] * 10 for _ in range(6)]
sensitivity_denominator_list = [[0] * 10 for _ in range(6)]
specificity_numerator_list = [[0] * 10 for _ in range(6)]
specificity_denominator_list = [[0] * 10 for _ in range(6)]
precision_numerator_list = [[0] * 10 for _ in range(6)]
precision_denominator_list = [[0] * 10 for _ in range(6)]

# Group the DataFrame by n_index, fold_index, and group_index
grouped = df.groupby(["n_index", "fold_index", "group_index"])

for (n_idx, fold, group_idx), group in grouped:

    std_ref_col = df.loc[group.index[1], result_cols] if len(group) > 1 else None

    true_count = (group.iloc[2][result_cols] == "True").sum()
    false_count = (group.iloc[2][result_cols] == "False").sum()
    accuracy_numerator_list[int(n_idx)][int(fold)] += true_count
    accuracy_denominator_list[int(n_idx)][int(fold)] += true_count + false_count

    sensitivity_numerator, sensitivity_denominator = sensitivity_calculator(group, result_cols)
    sensitivity_numerator_list[int(n_idx)][int(fold)] += sensitivity_numerator
    sensitivity_denominator_list[int(n_idx)][int(fold)] += sensitivity_denominator

    specificity_numerator, specificity_denominator = specificity_calculator(group, result_cols)
    specificity_numerator_list[int(n_idx)][int(fold)] += specificity_numerator
    specificity_denominator_list[int(n_idx)][int(fold)] += specificity_denominator

    precision_numerator, precision_denominator = precision_calculator(group, result_cols)
    precision_numerator_list[int(n_idx)][int(fold)] += precision_numerator
    precision_denominator_list[int(n_idx)][int(fold)] += precision_denominator

results_numeric_for_fold = result_calculator_for_each(5,5,accuracy_numerator_list, accuracy_denominator_list, sensitivity_numerator_list, sensitivity_denominator_list, specificity_numerator_list, specificity_denominator_list, precision_numerator_list, precision_denominator_list)
result_numeric_all = result_calculator_all(5,5,results_numeric_for_fold)

#Store the numeric results in a separate variable.
accuracy_numeric_numerator_list = copy.deepcopy(accuracy_numerator_list)
accuracy_numeric_denominator_list = copy.deepcopy(accuracy_denominator_list)
sensitivity_numeric_numerator_list = copy.deepcopy(sensitivity_numerator_list)
sensitivity_numeric_denominator_list = copy.deepcopy(sensitivity_denominator_list)
specificity_numeric_numerator_list = copy.deepcopy(specificity_numerator_list)
specificity_numeric_denominator_list = copy.deepcopy(specificity_denominator_list)
precision_numeric_numerator_list = copy.deepcopy(precision_numerator_list)
precision_numeric_denominator_list = copy.deepcopy(precision_denominator_list)

## string

In [ ]:
from pickle import TRUE
import pandas as pd
import numpy as np

# Load the Excel file (first sheet named "integrate")
file_name = "analysis_string_after_modification"
file_path = file_name
df = pd.read_excel(file_name, sheet_name="integrate")

# Use only the first 13 columns and reset index
df = df.iloc[:, :13].reset_index(drop=True)

# Initialize lists to store results for each metric across 6 n_index values and 10 folds
accuracy_numerator_list = [[0] * 10 for _ in range(6)]
accuracy_denominator_list = [[0] * 10 for _ in range(6)]
sensitivity_numerator_list = [[0] * 10 for _ in range(6)]
sensitivity_denominator_list = [[0] * 10 for _ in range(6)]
specificity_numerator_list = [[0] * 10 for _ in range(6)]
specificity_denominator_list = [[0] * 10 for _ in range(6)]
precision_numerator_list = [[0] * 10 for _ in range(6)]
precision_denominator_list = [[0] * 10 for _ in range(6)]

# Group the data by n_index and fold_index for fold-wise evaluation
grouped = df.groupby(["n_index", "fold_index"])

for (n_idx, fold), group in grouped:
    for index, row in group.iterrows():
        # Skip rows where the variable is "arm_name"
        if row["variable_name"]=="arm_name":
          continue;

        # Accuracy: all non-skipped variables are counted
        accuracy_denominator_list[int(n_idx)][int(fold)] += 1
        if int(row["FINAL"]) == 1:
            accuracy_numerator_list[int(n_idx)][int(fold)] += 1

        # Sensitivity: when human has extracted a value
        if row["human"] not in ["", "*", "NaN", np.nan]:
            sensitivity_denominator_list[int(n_idx)][int(fold)] += 1
            if int(row["FINAL"]) == 1:
                sensitivity_numerator_list[int(n_idx)][int(fold)] += 1

        # Specificity: when human did not extract a value
        elif row["human"] in ["", "*", "NaN", np.nan]:
            specificity_denominator_list[int(n_idx)][int(fold)] += 1
            if int(row["FINAL"]) == 1:
                specificity_numerator_list[int(n_idx)][int(fold)] += 1

        # precision: only when gpt extracted a value
        if not(str(row["gpt"])=="-1" or (str(row["gpt"]) == "null")):
            precision_denominator_list[int(n_idx)][int(fold)] += 1
            if int(row["FINAL"]) == 1:
                precision_numerator_list[int(n_idx)][int(fold)] += 1

results_string_for_fold = result_calculator_for_each(5,5,accuracy_numerator_list, accuracy_denominator_list, sensitivity_numerator_list, sensitivity_denominator_list, specificity_numerator_list, specificity_denominator_list, precision_numerator_list, precision_denominator_list)
result_string_all = result_calculator_all(5,5,results_string_for_fold)

# Store the raw numeric results separately (deep copy for preservation)
accuracy_string_numerator_list = copy.deepcopy(accuracy_numerator_list)
accuracy_string_denominator_list = copy.deepcopy(accuracy_denominator_list)
sensitivity_string_numerator_list = copy.deepcopy(sensitivity_numerator_list)
sensitivity_string_denominator_list = copy.deepcopy(sensitivity_denominator_list)
specificity_string_numerator_list = copy.deepcopy(specificity_numerator_list)
specificity_string_denominator_list = copy.deepcopy(specificity_denominator_list)
precision_string_numerator_list = copy.deepcopy(precision_numerator_list)
precision_string_denominator_list = copy.deepcopy(precision_denominator_list)

## all

In [ ]:
for n_idx in range(5,6):
    for fold in range(10):
        # Integrate the results of numeric and string variables
        accuracy_numerator_list[n_idx][fold] = accuracy_numeric_numerator_list[n_idx][fold] + accuracy_string_numerator_list[n_idx][fold]
        accuracy_denominator_list[n_idx][fold] = accuracy_numeric_denominator_list[n_idx][fold] + accuracy_string_denominator_list[n_idx][fold]
        sensitivity_numerator_list[n_idx][fold] = sensitivity_numeric_numerator_list[n_idx][fold] + sensitivity_string_numerator_list[n_idx][fold]
        sensitivity_denominator_list[n_idx][fold] = sensitivity_numeric_denominator_list[n_idx][fold] + sensitivity_string_denominator_list[n_idx][fold]
        specificity_numerator_list[n_idx][fold] = specificity_numeric_numerator_list[n_idx][fold] + specificity_string_numerator_list[n_idx][fold]
        specificity_denominator_list[n_idx][fold] = specificity_numeric_denominator_list[n_idx][fold] + specificity_string_denominator_list[n_idx][fold]
        precision_numerator_list[n_idx][fold] = precision_numeric_numerator_list[n_idx][fold] + precision_string_numerator_list[n_idx][fold]
        precision_denominator_list[n_idx][fold] = precision_numeric_denominator_list[n_idx][fold] + precision_string_denominator_list[n_idx][fold]

results_for_fold = result_calculator_for_each(5,5,accuracy_numerator_list, accuracy_denominator_list, sensitivity_numerator_list, sensitivity_denominator_list, specificity_numerator_list, specificity_denominator_list, precision_numerator_list, precision_denominator_list)
result_all = result_calculator_all(5,5,results_for_fold)

## numeric variables

In [ ]:
from pickle import TRUE
import pandas as pd
import numpy as np
import copy

# Load Excel file (first sheet)
file_name = "analysis_contextual_chat_prompting_after_modification"
file_path = file_name + ".xlsx"
df = pd.read_excel(file_path, sheet_name=0)
df = df[:303].reset_index(drop=True)

# Fill missing values in n_index and fold_index
df[["n_index", "fold_index"]] = df[["n_index", "fold_index"]].fillna(method="ffill")

# Assign group index every 4 rows (for each evaluation group)
df["group_index"] = df.index // 4

# Identify the columns containing True/False results
result_cols = df.iloc[:, 6:-1].columns
print(result_cols)

# Initialize empty lists to store metric values
# Dimensions: [n_index (0-5)][fold_index (0-9)][number of variables]
num_cols = len(result_cols)
accuracy_numerator_list = [[[0] * num_cols for _ in range(10)] for _ in range(6)]
accuracy_denominator_list = [[[0] * num_cols for _ in range(10)] for _ in range(6)]
sensitivity_numerator_list = [[[0] * num_cols for _ in range(10)] for _ in range(6)]
sensitivity_denominator_list = [[[0] * num_cols for _ in range(10)] for _ in range(6)]
specificity_numerator_list = [[[0] * num_cols for _ in range(10)] for _ in range(6)]
specificity_denominator_list = [[[0] * num_cols for _ in range(10)] for _ in range(6)]
precision_numerator_list = [[[0] * num_cols for _ in range(10)] for _ in range(6)]
precision_denominator_list = [[[0] * num_cols for _ in range(10)] for _ in range(6)]

# Group the data by n_index, fold_index, and group_index (every 4 rows)
grouped = df.groupby(["n_index", "fold_index", "group_index"])

for (n_idx, fold, group_idx), group in grouped:
    # Standard_Ref の列を特定（2番目の行のG~BL列を取得）
    std_ref_col = df.loc[group.index[1], result_cols] if len(group) > 1 else None

    # Compute Accuracy per variable
    for i, col in enumerate(result_cols):
        accuracy_denominator_list[int(n_idx)][int(fold)][i] += 1
        if group.iloc[2][col] in ["NA", np.nan]:
            accuracy_denominator_list[int(n_idx)][int(fold)][i] -= 1
        elif group.iloc[2][col] == "True":
            accuracy_numerator_list[int(n_idx)][int(fold)][i] += 1

    # Compute Sensitivity
    sensitivity_numerator, sensitivity_denominator = sensitivity_calculator_variable(group, result_cols)
    for i, col in enumerate(result_cols):
        sensitivity_numerator_list[int(n_idx)][int(fold)][i] += sensitivity_numerator[i]
        sensitivity_denominator_list[int(n_idx)][int(fold)][i] += sensitivity_denominator[i]

    # Compute Specificity
    specificity_numerator, specificity_denominator = specificity_calculator_variable(group, result_cols)
    for i, col in enumerate(result_cols):
        specificity_numerator_list[int(n_idx)][int(fold)][i] += specificity_numerator[i]
        specificity_denominator_list[int(n_idx)][int(fold)][i] += specificity_denominator[i]

    # Compute precision
    precision_numerator, precision_denominator = precision_calculator_variable(group, result_cols)
    for i, col in enumerate(result_cols):
        precision_numerator_list[int(n_idx)][int(fold)][i] += precision_numerator[i]
        precision_denominator_list[int(n_idx)][int(fold)][i] += precision_denominator[i]

# Convert lists to NumPy arrays and rearrange axes to [variable][fold][n_index]
accuracy_numerator_list = np.array(accuracy_numerator_list)
accuracy_numerator_list = np.moveaxis(accuracy_numerator_list, [0, 1, 2], [1,2,0])
accuracy_numerator_list = accuracy_numerator_list.tolist()

accuracy_denominator_list = np.array(accuracy_denominator_list)
accuracy_denominator_list = np.moveaxis(accuracy_denominator_list, [0, 1, 2], [1,2,0])
accuracy_denominator_list = accuracy_denominator_list.tolist()

sensitivity_numerator_list = np.array(sensitivity_numerator_list)
sensitivity_numerator_list = np.moveaxis(sensitivity_numerator_list, [0, 1, 2], [1,2,0])
sensitivity_numerator_list = sensitivity_numerator_list.tolist()

sensitivity_denominator_list = np.array(sensitivity_denominator_list)
sensitivity_denominator_list = np.moveaxis(sensitivity_denominator_list, [0, 1, 2], [1,2,0])
sensitivity_denominator_list = sensitivity_denominator_list.tolist()

specificity_numerator_list = np.array(specificity_numerator_list)
specificity_numerator_list = np.moveaxis(specificity_numerator_list, [0, 1, 2], [1,2,0])
specificity_numerator_list = specificity_numerator_list.tolist()

specificity_denominator_list = np.array(specificity_denominator_list)
specificity_denominator_list = np.moveaxis(specificity_denominator_list, [0, 1, 2], [1,2,0])
specificity_denominator_list = specificity_denominator_list.tolist()

precision_numerator_list = np.array(precision_numerator_list)
precision_numerator_list = np.moveaxis(precision_numerator_list, [0, 1, 2], [1,2,0])

precision_denominator_list = np.array(precision_denominator_list)
precision_denominator_list = np.moveaxis(precision_denominator_list, [0, 1, 2], [1,2,0])

# Calculate metrics for each variable across folds
results_numeric_variables = {}
for i,col in enumerate(result_cols):
    # Calculate metrics per fold for this variable
    results_numeric_variable_for_fold = result_calculator_for_each(
        5, 5,
        accuracy_numerator_list[i], accuracy_denominator_list[i],
        sensitivity_numerator_list[i], sensitivity_denominator_list[i],
        specificity_numerator_list[i], specificity_denominator_list[i],
        precision_numerator_list[i], precision_denominator_list[i]
    )
    result_numeric_variable_all = result_calculator_all(5, 5, results_numeric_variable_for_fold)
    results_numeric_variables[col] =  result_numeric_variable_all

# Flatten the nested result dict into a list of records
flattened_data = []
for variable_name, records in results_numeric_variables.items():
    for record in records:
        record["variable_name"] = variable_name  # variable_name を追加
        flattened_data.append(record)

# Convert the flattened result into a DataFrame
df_numeric_varibles = pd.DataFrame(flattened_data)

df_numeric_varibles.set_index("variable_name", inplace=True)

<ipython-input-42-184accf4cfec>:13: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[["n_index", "fold_index"]] = df[["n_index", "fold_index"]].fillna(method="ffill")


Index(['Year', 'ICC_for_cRCT', 'Age_mean', 'Age_sd', 'Age_n', 'F_n', 'Hyp_n',
       'Primary_insomnia_n', 'se', 'sd', 'cr', 'cw', 'sr', 'sc', 're', 'pi',
       'w', 'dt', 'ns', 'he', 'tg', 'ind', 'gp', 'ff', 'ae', 'n', 'remission',
       'Severity_bl_mean', 'Severity_bl_sd', 'Severity_bl_n',
       'Severity_ch_mean', 'Severity_ch_sd', 'Severity_ch_n',
       'Severity_ep_mean', 'Severity_ep_sd', 'Severity_ep_n', 'dropout',
       'SE_mean', 'SE_sd', 'SE_n', 'TST_mean', 'TST_sd', 'TST_n', 'SOL_mean',
       'SOL_sd', 'SOL_n', 'WASO_mean', 'WASO_sd', 'WASO_n',
       'Sleep_diary_weeks', 'r_long', 'Severity_mean_long', 'Severity_sd_long',
       'Severity_n_long', 'Severity_ch_mean_long', 'Severity_ch_sd_long',
       'Severity_ch_n_long', 'r_weeks_long'],
      dtype='object')


## string variables

In [ ]:
from pickle import TRUE
import pandas as pd
import numpy as np

# Load the Excel file (first sheet named "integrate")
file_name = "analysis_string_after_modification"
file_path = file_name
df = pd.read_excel(file_name, sheet_name="integrate")
df = df.iloc[:, :13].reset_index(drop=True)

# Get the list of unique variable names, excluding "arm_name"
variable_list = df["variable_name"].unique().tolist()
variable_list.remove("arm_name")
print(variable_list)

# Initialize lists to store metric results
# Structure: [variable][n_index (0–5)][fold_index (0–9)]
accuracy_numerator_list = [[[0] * 10 for _ in range(6)] for _ in range(len(variable_list))]
accuracy_denominator_list = [[[0] * 10 for _ in range(6)] for _ in range(len(variable_list))]
sensitivity_numerator_list = [[[0] * 10 for _ in range(6)] for _ in range(len(variable_list))]
sensitivity_denominator_list = [[[0] * 10 for _ in range(6)] for _ in range(len(variable_list))]
specificity_numerator_list = [[[0] * 10 for _ in range(6)] for _ in range(len(variable_list))]
specificity_denominator_list = [[[0] * 10 for _ in range(6)] for _ in range(len(variable_list))]
precision_numerator_list = [[[0] * 10 for _ in range(6)] for _ in range(len(variable_list))]
precision_denominator_list = [[[0] * 10 for _ in range(6)] for _ in range(len(variable_list))]

# Group by n_index and fold_index
grouped = df.groupby(["n_index", "fold_index"])

# Compute each metric for every variable in each group
for (n_idx, fold), group in grouped:
    for index, row in group.iterrows():
        if row["variable_name"]=="arm_name":
          continue; # Skip if the variable is "arm_name"
        variable_idx = variable_list.index(row["variable_name"])

        # Accuracy: Count all valid rows
        accuracy_denominator_list[variable_idx][int(n_idx)][int(fold)] += 1
        if int(row["FINAL"]) == 1:
            accuracy_numerator_list[variable_idx][int(n_idx)][int(fold)] += 1

        # Sensitivity: when human provided an extraction
        if row["human"] not in ["", "*", "NaN", np.nan]:
            sensitivity_denominator_list[variable_idx][int(n_idx)][int(fold)] += 1
            if int(row["FINAL"]) == 1:
                sensitivity_numerator_list[variable_idx][int(n_idx)][int(fold)] += 1

        # Specificity: when human did NOT extract
        elif row["human"] in ["", "*", "NaN", np.nan]:
            specificity_denominator_list[variable_idx][int(n_idx)][int(fold)] += 1
            if int(row["FINAL"]) == 1:
                specificity_numerator_list[variable_idx][int(n_idx)][int(fold)] += 1

        # precision: only when gpt extracted
        if not(str(row["gpt"])=="-1" or (str(row["gpt"]) == "null")):
            precision_denominator_list[variable_idx][int(n_idx)][int(fold)] += 1
            if int(row["FINAL"]) == 1:
                precision_numerator_list[variable_idx][int(n_idx)][int(fold)] += 1


# Calculate metrics for each variable
results_string_variables = {}
for i,variable in enumerate(variable_list):
    results_string_variable_for_fold = result_calculator_for_each(
        5, 5,
        accuracy_numerator_list[i], accuracy_denominator_list[i],
        sensitivity_numerator_list[i], sensitivity_denominator_list[i],
        specificity_numerator_list[i], specificity_denominator_list[i],
        precision_numerator_list[i], precision_denominator_list[i]
    )
    result_string_variable_all = result_calculator_all(5, 5, results_string_variable_for_fold)
    results_string_variables[variable] = result_string_variable_all


# Flatten the results into a single list of dictionaries
flattened_data = []
for variable_name, records in results_string_variables.items():
    for record in records:
        record["variable_name"] = variable_name
        flattened_data.append(record)

# Convert the results into a DataFrame
df_string_varibles = pd.DataFrame(flattened_data)

df_string_varibles.set_index("variable_name", inplace=True)

['ind_clu', 'Country', 'Single_multi', 'Insomnia_diagnosis', 'Comorbidities', 'def_remission_response', 'scale_used', 'Severity_scale', 'r_long_scale_used']


In [ ]:
# Create DataFrames for fold-level and overall results
df_numeric_for_each = pd.DataFrame(results_numeric_for_fold) # Numeric data - per fold
df_string_for_each = pd.DataFrame(results_string_for_fold) # String data - per fold
df_for_each = pd.DataFrame(results_for_fold) # Overall result - per fold

df_numeric_all = pd.DataFrame(result_numeric_all) # Numeric data - averaged across folds
df_string_all = pd.DataFrame(result_string_all) # String data - averaged across folds
df_all = pd.DataFrame(result_all) # Overall result - averaged across folds

# Save all DataFrames into a single Excel file, each on a separate sheet
with pd.ExcelWriter("new_sensitivity_specificity/" + file_name+"_new_sensitivity_specificity.xlsx") as writer:
    df_numeric_for_each.to_excel(writer, sheet_name="numeric_for_each", index=False)
    df_string_for_each.to_excel(writer, sheet_name="string_for_each", index=False)
    df_for_each.to_excel(writer, sheet_name="result_for_each", index=False)
    df_numeric_all.to_excel(writer, sheet_name="numeric_all", index=False)
    df_string_all.to_excel(writer, sheet_name="string_all", index=False)
    df_all.to_excel(writer, sheet_name="result_all", index=False)
    df_numeric_varibles.to_excel(writer, sheet_name="numeric_variables", index=True)
    df_string_varibles.to_excel(writer, sheet_name="string_variables", index=True)